In [1]:
# ============================================================
# CELL 1 — IMPORTS
# EfficientNet-B0 Mouth ROI Deepfake Detection
# ============================================================

import copy
import hashlib
import json
import os
import random
import shutil
import time
import warnings

from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import (
    EfficientNet_B0_Weights,
    efficientnet_b0,
)
from tqdm.auto import tqdm


warnings.filterwarnings("default")

print("=" * 80)
print("IMPORTS PASSED")
print("=" * 80)

print("Python environment is ready.")
print("PyTorch version    :", torch.__version__)
print("CUDA available     :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device        :", torch.cuda.get_device_name(0))
else:
    print("CUDA device        : CPU")

IMPORTS PASSED
Python environment is ready.
PyTorch version    : 2.11.0+cpu
CUDA available     : False
CUDA device        : CPU


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# EfficientNet-B0 Mouth ROI Deepfake Detection
# ============================================================

RUN_SCHEMA_VERSION = 4


@dataclass(frozen=True)
class Config:
    # Reproducibility
    seed: int = 42

    # Image settings
    image_size: int = 224

    # DataLoader settings
    batch_size: int = 32
    num_workers: int = 0

    # Two-stage transfer learning
    frozen_epochs: int = 5
    finetune_epochs: int = 15

    # Learning rates
    frozen_lr: float = 1e-3
    finetune_lr: float = 2e-5

    # Optimization
    weight_decay: float = 1e-4
    early_stopping_patience: int = 4
    scheduler_patience: int = 2
    grad_clip_norm: float = 1.0
    dropout: float = 0.2

    # Validation threshold search
    threshold_min: float = 0.05
    threshold_max: float = 0.95
    threshold_steps: int = 181

    # Dataset labels and accepted ROI statuses
    accepted_statuses: Tuple[str, ...] = ("ok", "success")
    negative_label: str = "real"
    positive_label: str = "fake"

    # Local Colab SSD cache
    cache_images_locally: bool = True
    verify_cached_images: bool = True

    # Resume only a compatible incomplete mouth run
    resume_compatible_run: bool = True


CONFIG = Config()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("CONFIGURATION PASSED")
print("=" * 80)

print(CONFIG)
print()
print("DEVICE:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print(
        "NOTE: CPU is currently active. "
        "GPU will be required before the training stages."
    )

CONFIGURATION PASSED
Config(seed=42, image_size=224, batch_size=32, num_workers=0, frozen_epochs=5, finetune_epochs=15, frozen_lr=0.001, finetune_lr=2e-05, weight_decay=0.0001, early_stopping_patience=4, scheduler_patience=2, grad_clip_norm=1.0, dropout=0.2, threshold_min=0.05, threshold_max=0.95, threshold_steps=181, accepted_statuses=('ok', 'success'), negative_label='real', positive_label='fake', cache_images_locally=True, verify_cached_images=True, resume_compatible_run=True)

DEVICE: cpu
NOTE: CPU is currently active. GPU will be required before the training stages.


In [3]:
# ============================================================
# CELL 3 — REPRODUCIBILITY
# ============================================================

def seed_everything(seed: int) -> None:
    # Python random
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch CPU
    torch.manual_seed(seed)

    # PyTorch CUDA
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Deterministic CUDA behaviour
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Reproducible Python hashing
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(CONFIG.seed)

# Separate generator for reproducible train DataLoader shuffling
TRAIN_GENERATOR = torch.Generator()
TRAIN_GENERATOR.manual_seed(CONFIG.seed)

print("=" * 80)
print("REPRODUCIBILITY PASSED")
print("=" * 80)

print("Global seed          :", CONFIG.seed)
print("Python random seeded : YES")
print("NumPy seeded         : YES")
print("PyTorch seeded       : YES")
print("Train generator ready: YES")

REPRODUCIBILITY PASSED
Global seed          : 42
Python random seeded : YES
NumPy seeded         : YES
PyTorch seeded       : YES
Train generator ready: YES


In [4]:
# ============================================================
# CELL 4 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

DRIVE_MOUNT_POINT = Path("/content/drive")

drive.mount(
    str(DRIVE_MOUNT_POINT),
    force_remount=False,
)

MY_DRIVE = DRIVE_MOUNT_POINT / "MyDrive"

if not MY_DRIVE.is_dir():
    raise FileNotFoundError(
        f"Google Drive MyDrive directory was not found:\n{MY_DRIVE}"
    )

print("=" * 80)
print("GOOGLE DRIVE MOUNT PASSED")
print("=" * 80)

print("Drive mount point:", DRIVE_MOUNT_POINT)
print("MyDrive path     :", MY_DRIVE)

Mounted at /content/drive
GOOGLE DRIVE MOUNT PASSED
Drive mount point: /content/drive
MyDrive path     : /content/drive/MyDrive


In [10]:
# ============================================================
# CELL 5 — AUTOMATIC MOUTH ROI PATH DISCOVERY
# No Turkish folder names are hard-coded
# ============================================================

# Find the real AISC project directory.
aisc_candidates = sorted(
    [
        path
        for path in MY_DRIVE.iterdir()
        if path.is_dir()
        and path.name.strip().lower().startswith("aisc")
    ],
    key=lambda path: path.name,
)

if not aisc_candidates:
    raise FileNotFoundError(
        "No directory starting with 'AISC' was found under MyDrive."
    )

print("AISC candidates found:")
for candidate in aisc_candidates:
    print(" -", candidate)


# Search for metadata.csv whose parent directory is mouth_roi_output.
metadata_candidates = []

for aisc_root in aisc_candidates:
    for metadata_file in aisc_root.rglob("metadata.csv"):
        if metadata_file.parent.name == "mouth_roi_output":
            metadata_candidates.append(
                metadata_file
            )


if not metadata_candidates:
    raise FileNotFoundError(
        "No metadata.csv file inside a mouth_roi_output "
        "directory was found under the AISC folders."
    )


print()
print("Mouth metadata candidates found:")

for index, candidate in enumerate(
    metadata_candidates,
    start=1,
):
    print(f" {index}. {candidate}")


# Prefer the path belonging to Dilara / Deney 1.
preferred_candidates = [
    path
    for path in metadata_candidates
    if "Dilara" in path.parts
    and "Deney 1" in path.parts
]

if len(preferred_candidates) == 1:
    METADATA_PATH = preferred_candidates[0]

elif len(metadata_candidates) == 1:
    METADATA_PATH = metadata_candidates[0]

else:
    raise RuntimeError(
        "More than one suitable mouth metadata.csv was found. "
        "Please send the candidate list printed above."
    )


ROI_ROOT = METADATA_PATH.parent
MOUTH_ROOT = ROI_ROOT.parent
DENEY1_ROOT = MOUTH_ROOT.parent


# Locate the results directory without writing Turkish characters.
results_candidates = [
    path
    for path in DENEY1_ROOT.iterdir()
    if path.is_dir()
    and path.name.casefold().startswith("sonu")
]

if len(results_candidates) != 1:
    raise RuntimeError(
        "The results directory could not be selected uniquely.\n"
        f"Candidates found: {results_candidates}"
    )

RESULTS_ROOT = results_candidates[0]


print()
print("=" * 80)
print("MOUTH ROI PATH CONFIGURATION PASSED")
print("=" * 80)

print("DENEY1_ROOT :", DENEY1_ROOT)
print("MOUTH_ROOT  :", MOUTH_ROOT)
print("ROI_ROOT    :", ROI_ROOT)
print("METADATA    :", METADATA_PATH)
print("RESULTS_ROOT:", RESULTS_ROOT)

AISC candidates found:
 - /content/drive/MyDrive/AISC Çalışmalar


/usr/lib/python3.12/pathlib.py:1063: ResourceWarning: unclosed scandir iterator <posix.ScandirIterator object at 0x7af913ffc630>
  return os.scandir(self)


KeyboardInterrupt: 

In [11]:
# ============================================================
# CELL 5 — FAST MOUTH ROI PATH DISCOVERY
# ============================================================

# 1. Find the single AISC directory under MyDrive.
aisc_candidates = [
    path
    for path in MY_DRIVE.iterdir()
    if path.is_dir()
    and path.name.lower().startswith("aisc")
]

if len(aisc_candidates) != 1:
    raise RuntimeError(
        "AISC directory could not be selected uniquely.\n"
        f"Candidates: {aisc_candidates}"
    )

PROJECT_ROOT = aisc_candidates[0]


# 2. These folder names contain no problematic Turkish characters.
EXPERIMENTS_ROOT = PROJECT_ROOT / "Deneyler"
DILARA_ROOT = EXPERIMENTS_ROOT / "Dilara"
DENEY1_ROOT = DILARA_ROOT / "Deney 1"

if not DENEY1_ROOT.is_dir():
    raise FileNotFoundError(
        f"Deney 1 directory was not found:\n{DENEY1_ROOT}"
    )


# 3. Find mouth_roi_output one level below the region folders.
roi_candidates = []

for region_directory in DENEY1_ROOT.iterdir():
    if not region_directory.is_dir():
        continue

    candidate = (
        region_directory
        / "mouth_roi_output"
    )

    if candidate.is_dir():
        roi_candidates.append(candidate)


if len(roi_candidates) != 1:
    raise RuntimeError(
        "mouth_roi_output could not be selected uniquely.\n"
        f"Candidates: {roi_candidates}"
    )

ROI_ROOT = roi_candidates[0]
MOUTH_ROOT = ROI_ROOT.parent
METADATA_PATH = ROI_ROOT / "metadata.csv"


if not METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"metadata.csv was not found:\n{METADATA_PATH}"
    )


# 4. Find the results directory using its ASCII prefix.
results_candidates = [
    path
    for path in DENEY1_ROOT.iterdir()
    if path.is_dir()
    and path.name.lower().startswith("sonu")
]


if len(results_candidates) != 1:
    raise RuntimeError(
        "Results directory could not be selected uniquely.\n"
        f"Candidates: {results_candidates}"
    )

RESULTS_ROOT = results_candidates[0]


print("=" * 80)
print("MOUTH ROI PATH CONFIGURATION PASSED")
print("=" * 80)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DENEY1_ROOT  :", DENEY1_ROOT)
print("MOUTH_ROOT   :", MOUTH_ROOT)
print("ROI_ROOT     :", ROI_ROOT)
print("METADATA     :", METADATA_PATH)
print("RESULTS_ROOT :", RESULTS_ROOT)

MOUTH ROI PATH CONFIGURATION PASSED
PROJECT_ROOT : /content/drive/MyDrive/AISC Çalışmalar
DENEY1_ROOT  : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1
MOUTH_ROOT   : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Ağız
ROI_ROOT     : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output
METADATA     : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output/metadata.csv
RESULTS_ROOT : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar


In [12]:
# ============================================================
# CELL 6 — RUN DIRECTORY + SAFE RESUME POLICY
# ============================================================

RUN_SUFFIX = "mouth_efficientnet_b0_seed42"

RUN_SCHEMA_FILE = "run_schema.json"
RUN_COMPLETE_FILE = "RUN_COMPLETE.json"


def is_compatible_incomplete_run(
    path: Path,
) -> bool:
    """
    A run can be resumed only if:
    - it is a directory,
    - it is not marked as complete,
    - it contains a schema file,
    - its schema version matches this notebook.
    """
    schema_path = (
        path
        / RUN_SCHEMA_FILE
    )

    complete_path = (
        path
        / RUN_COMPLETE_FILE
    )

    if not path.is_dir():
        return False

    if complete_path.exists():
        return False

    if not schema_path.is_file():
        return False

    try:
        with open(
            schema_path,
            "r",
            encoding="utf-8",
        ) as file:
            schema = json.load(file)

        return (
            int(
                schema.get(
                    "schema_version",
                    -1,
                )
            )
            == RUN_SCHEMA_VERSION
        )

    except Exception:
        return False


compatible_runs = []

if CONFIG.resume_compatible_run:
    for candidate in RESULTS_ROOT.glob(
        f"*_{RUN_SUFFIX}*"
    ):
        if is_compatible_incomplete_run(
            candidate
        ):
            compatible_runs.append(
                candidate
            )


compatible_runs.sort(
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)


if compatible_runs:
    RUN_DIR = compatible_runs[0]
    RUN_ID = RUN_DIR.name

    print(
        "Resuming compatible incomplete mouth run:"
    )
    print(RUN_DIR)

else:
    RUN_ID = (
        datetime.now().strftime(
            "%Y%m%d_%H%M"
        )
        + f"_{RUN_SUFFIX}"
    )

    RUN_DIR = (
        RESULTS_ROOT
        / RUN_ID
    )

    retry = 1

    while RUN_DIR.exists():
        RUN_DIR = (
            RESULTS_ROOT
            / f"{RUN_ID}_r{retry}"
        )

        retry += 1

    RUN_ID = RUN_DIR.name

    RUN_DIR.mkdir(
        parents=True,
        exist_ok=False,
    )

    with open(
        RUN_DIR / RUN_SCHEMA_FILE,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            {
                "schema_version": RUN_SCHEMA_VERSION,
                "run_id": RUN_ID,
                "experiment": (
                    "EfficientNet-B0 Mouth ROI "
                    "Transfer Learning Baseline"
                ),
                "seed": CONFIG.seed,
                "created_at": datetime.now().isoformat(),
            },
            file,
            indent=2,
            ensure_ascii=False,
        )


DIRS = {
    "checkpoints": RUN_DIR / "checkpoints",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "artifacts": RUN_DIR / "artifacts",
    "logs": RUN_DIR / "logs",
}


for directory in DIRS.values():
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


FROZEN_DIR = (
    DIRS["checkpoints"]
    / "frozen"
)

FINETUNE_DIR = (
    DIRS["checkpoints"]
    / "finetune"
)

FROZEN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINETUNE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("=" * 80)
print("RUN DIRECTORY CONFIGURATION PASSED")
print("=" * 80)

print("RUN_ID      :", RUN_ID)
print("RUN_DIR     :", RUN_DIR)
print("FROZEN_DIR  :", FROZEN_DIR)
print("FINETUNE_DIR:", FINETUNE_DIR)

RUN DIRECTORY CONFIGURATION PASSED
RUN_ID      : 20260808_1257_mouth_efficientnet_b0_seed42
RUN_DIR     : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar/20260808_1257_mouth_efficientnet_b0_seed42
FROZEN_DIR  : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar/20260808_1257_mouth_efficientnet_b0_seed42/checkpoints/frozen
FINETUNE_DIR: /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar/20260808_1257_mouth_efficientnet_b0_seed42/checkpoints/finetune


In [13]:
# ============================================================
# CELL 7 — ATOMIC I/O
# ============================================================

import platform
import subprocess
import sys
from typing import Any

import yaml


def atomic_write_bytes(
    data: bytes,
    target: Path,
) -> None:
    """
    Write bytes to a temporary file first, then replace
    the target atomically.
    """
    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = target.with_suffix(
        target.suffix + ".tmp"
    )

    with open(
        temporary_path,
        "wb",
    ) as file:
        file.write(data)
        file.flush()
        os.fsync(file.fileno())

    os.replace(
        temporary_path,
        target,
    )


def atomic_write_text(
    text: str,
    target: Path,
    encoding: str = "utf-8",
) -> None:
    atomic_write_bytes(
        text.encode(encoding),
        target,
    )


def atomic_json_dump(
    obj: Any,
    target: Path,
) -> None:
    atomic_write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        target,
    )


def atomic_yaml_dump(
    obj: Any,
    target: Path,
) -> None:
    atomic_write_text(
        yaml.safe_dump(
            obj,
            sort_keys=False,
            allow_unicode=True,
        ),
        target,
    )


def atomic_csv_dump(
    dataframe: pd.DataFrame,
    target: Path,
) -> None:
    atomic_write_bytes(
        dataframe.to_csv(
            index=False
        ).encode("utf-8"),
        target,
    )


def atomic_torch_save(
    state: dict,
    target: Path,
) -> None:
    """
    Save a checkpoint temporarily, reload it on CPU,
    validate its schema, and only then publish it.
    """
    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = target.with_suffix(
        target.suffix + ".tmp"
    )

    torch.save(
        state,
        temporary_path,
    )

    # Read-back checkpoint validation on CPU
    verification = torch.load(
        temporary_path,
        map_location="cpu",
        weights_only=False,
    )

    required_keys = {
        "checkpoint_schema_version",
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
        "stage_name",
    }

    if not required_keys.issubset(
        verification.keys()
    ):
        try:
            temporary_path.unlink()
        finally:
            raise RuntimeError(
                "Checkpoint validation failed: "
                f"{temporary_path}"
            )

    os.replace(
        temporary_path,
        target,
    )


def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    """
    Calculate a SHA-256 hash for output integrity checks.
    """
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# Save resolved experiment configuration.
atomic_yaml_dump(
    asdict(CONFIG),
    RUN_DIR / "config_resolved.yaml",
)


# Save runtime environment information.
atomic_json_dump(
    {
        "run_id": RUN_ID,
        "schema_version": RUN_SCHEMA_VERSION,
        "experiment": (
            "EfficientNet-B0 Mouth ROI "
            "Transfer Learning Baseline"
        ),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "device": str(DEVICE),
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
        "platform": platform.platform(),
    },
    RUN_DIR / "environment.json",
)


# Save installed Python dependency versions.
try:
    requirements_freeze = subprocess.check_output(
        [
            sys.executable,
            "-m",
            "pip",
            "freeze",
        ],
        text=True,
    )

    atomic_write_text(
        requirements_freeze,
        RUN_DIR / "requirements_lock.txt",
    )

except Exception as error:
    atomic_write_text(
        f"pip freeze failed: {repr(error)}\n",
        RUN_DIR / "requirements_lock.txt",
    )


print("=" * 80)
print("ATOMIC I/O PASSED")
print("=" * 80)

print("Atomic I/O helpers ready.")
print("Configuration saved :", RUN_DIR / "config_resolved.yaml")
print("Environment saved   :", RUN_DIR / "environment.json")
print("Requirements saved  :", RUN_DIR / "requirements_lock.txt")

ATOMIC I/O PASSED
Atomic I/O helpers ready.
Configuration saved : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar/20260808_1257_mouth_efficientnet_b0_seed42/config_resolved.yaml
Environment saved   : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar/20260808_1257_mouth_efficientnet_b0_seed42/environment.json
Requirements saved  : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar/20260808_1257_mouth_efficientnet_b0_seed42/requirements_lock.txt


In [14]:
# ============================================================
# CELL 8 — LOAD + VALIDATE MOUTH ROI METADATA
# ============================================================

REQUIRED_METADATA_COLUMNS = {
    "sample_id",
    "label",
    "split",
    "status",
    "mouth_path",
}

metadata_raw = pd.read_csv(
    METADATA_PATH,
    encoding="utf-8-sig",
)

metadata_raw.columns = [
    str(column).strip()
    for column in metadata_raw.columns
]


# ------------------------------------------------------------
# 1. Required column validation
# ------------------------------------------------------------

missing_columns = (
    REQUIRED_METADATA_COLUMNS
    .difference(metadata_raw.columns)
)

if missing_columns:
    raise ValueError(
        "metadata.csv is missing required columns: "
        f"{sorted(missing_columns)}"
    )


metadata = metadata_raw.copy()


# ------------------------------------------------------------
# 2. Clean text columns without converting missing values
#    into the literal string "nan"
# ------------------------------------------------------------

text_columns = [
    "sample_id",
    "label",
    "split",
    "status",
    "mouth_path",
]

optional_text_columns = [
    "video_id",
    "source_frame",
    "relative_frame_path",
    "frame_stem",
    "face_id",
]

for column in (
    text_columns
    + optional_text_columns
):
    if column in metadata.columns:
        metadata[column] = (
            metadata[column]
            .astype("string")
            .str.strip()
        )


metadata["label"] = (
    metadata["label"]
    .str.lower()
)

metadata["split"] = (
    metadata["split"]
    .str.lower()
)

metadata["status"] = (
    metadata["status"]
    .str.lower()
)


# ------------------------------------------------------------
# 3. Allowed values
# ------------------------------------------------------------

allowed_labels = {
    CONFIG.negative_label,
    CONFIG.positive_label,
}

allowed_splits = {
    "train",
    "val",
    "test",
}

accepted_statuses = {
    str(value).lower()
    for value in CONFIG.accepted_statuses
}


observed_labels = set(
    metadata["label"]
    .dropna()
    .unique()
)

observed_splits = set(
    metadata["split"]
    .dropna()
    .unique()
)


bad_labels = sorted(
    observed_labels
    - allowed_labels
)

bad_splits = sorted(
    observed_splits
    - allowed_splits
)


if bad_labels:
    raise ValueError(
        f"Unexpected labels: {bad_labels}"
    )

if bad_splits:
    raise ValueError(
        f"Unexpected splits: {bad_splits}"
    )


# ------------------------------------------------------------
# 4. Accepted and audit-only rows
# ------------------------------------------------------------

status_counts = (
    metadata["status"]
    .fillna("<missing>")
    .value_counts(dropna=False)
)

accepted_mask = (
    metadata["status"]
    .isin(accepted_statuses)
)

audit_only = (
    metadata.loc[
        ~accepted_mask
    ]
    .copy()
)

accepted = (
    metadata.loc[
        accepted_mask
    ]
    .copy()
)


if accepted.empty:
    raise RuntimeError(
        "No accepted mouth ROI rows exist."
    )


# ------------------------------------------------------------
# 5. Required-value validation
# ------------------------------------------------------------

missing_sample_mask = (
    accepted["sample_id"].isna()
    | accepted["sample_id"].eq("")
)

if missing_sample_mask.any():
    raise ValueError(
        "Accepted rows with missing sample_id were found."
    )


duplicate_sample_mask = (
    accepted["sample_id"]
    .duplicated(keep=False)
)

if duplicate_sample_mask.any():
    duplicate_examples = (
        accepted.loc[
            duplicate_sample_mask,
            "sample_id",
        ]
        .head(20)
        .tolist()
    )

    raise ValueError(
        "Duplicate sample_id values were found. "
        f"Examples: {duplicate_examples}"
    )


missing_path_mask = (
    accepted["mouth_path"].isna()
    | accepted["mouth_path"].eq("")
)

if missing_path_mask.any():
    bad_rows = accepted.loc[
        missing_path_mask,
        [
            "sample_id",
            "label",
            "split",
            "status",
        ],
    ].head(20)

    raise ValueError(
        "Accepted rows with missing mouth_path:\n"
        f"{bad_rows}"
    )


# ------------------------------------------------------------
# 6. Resolve mouth ROI paths
# ------------------------------------------------------------

def resolve_mouth_path(
    relative_or_absolute_path: str,
) -> Path:
    path = Path(
        str(relative_or_absolute_path)
    )

    if path.is_absolute():
        return path

    return (
        ROI_ROOT
        / path
    )


accepted[
    "resolved_mouth_path"
] = (
    accepted["mouth_path"]
    .map(resolve_mouth_path)
)


accepted[
    "path_exists"
] = (
    accepted["resolved_mouth_path"]
    .map(
        lambda path: path.is_file()
    )
)


missing_file_rows = (
    accepted.loc[
        ~accepted["path_exists"]
    ]
    .copy()
)


if not missing_file_rows.empty:
    missing_examples = (
        missing_file_rows[
            [
                "sample_id",
                "label",
                "split",
                "mouth_path",
                "resolved_mouth_path",
            ]
        ]
        .head(20)
    )

    raise FileNotFoundError(
        "Accepted mouth ROI files are missing:\n"
        f"{missing_examples}"
    )


# ------------------------------------------------------------
# 7. Final eligible dataset
# ------------------------------------------------------------

eligible = (
    accepted.loc[
        accepted["path_exists"]
    ]
    .copy()
    .reset_index(drop=True)
)


if eligible.empty:
    raise RuntimeError(
        "No training-eligible mouth ROI rows remain."
    )


# ------------------------------------------------------------
# 8. Every split must contain both classes
# ------------------------------------------------------------

split_class_counts = pd.crosstab(
    eligible["split"],
    eligible["label"],
)


for required_split in [
    "train",
    "val",
    "test",
]:
    if required_split not in split_class_counts.index:
        raise ValueError(
            f"Missing split: {required_split}"
        )

    split_labels = set(
        eligible.loc[
            eligible["split"]
            == required_split,
            "label",
        ].unique()
    )

    if split_labels != allowed_labels:
        raise ValueError(
            f"Split '{required_split}' does not contain "
            f"both classes: {split_labels}"
        )


# ------------------------------------------------------------
# 9. Mouth ROI path leakage validation
# ------------------------------------------------------------

split_path_sets = {
    split_name: set(
        eligible.loc[
            eligible["split"]
            == split_name,
            "resolved_mouth_path",
        ].map(str)
    )
    for split_name in [
        "train",
        "val",
        "test",
    ]
}


path_intersections = {
    "train_val": len(
        split_path_sets["train"]
        & split_path_sets["val"]
    ),
    "train_test": len(
        split_path_sets["train"]
        & split_path_sets["test"]
    ),
    "val_test": len(
        split_path_sets["val"]
        & split_path_sets["test"]
    ),
}


if any(
    value > 0
    for value in path_intersections.values()
):
    raise RuntimeError(
        "Cross-split mouth ROI path leakage detected: "
        f"{path_intersections}"
    )


# ------------------------------------------------------------
# 10. Video ID intersection inspection
# ------------------------------------------------------------

video_intersections = {}
video_leakage_status = (
    "NOT_VERIFIABLE_FROM_CURRENT_METADATA"
)

if (
    "video_id" in eligible.columns
    and eligible["video_id"].notna().any()
):
    split_video_sets = {
        split_name: set(
            eligible.loc[
                eligible["split"]
                == split_name,
                "video_id",
            ]
            .dropna()
            .astype(str)
        )
        for split_name in [
            "train",
            "val",
            "test",
        ]
    }

    video_intersections = {
        "train_val": len(
            split_video_sets["train"]
            & split_video_sets["val"]
        ),
        "train_test": len(
            split_video_sets["train"]
            & split_video_sets["test"]
        ),
        "val_test": len(
            split_video_sets["val"]
            & split_video_sets["test"]
        ),
    }

    if all(
        value == 0
        for value in video_intersections.values()
    ):
        video_leakage_status = (
            "ZERO_INTERSECTION_FOR_METADATA_VIDEO_ID"
        )
    else:
        video_leakage_status = (
            "INTERSECTION_DETECTED_FOR_METADATA_VIDEO_ID"
        )


# ------------------------------------------------------------
# 11. Save audit and accounting files
# ------------------------------------------------------------

atomic_csv_dump(
    audit_only,
    DIRS["artifacts"]
    / "audit_only_metadata_rows.csv",
)

atomic_csv_dump(
    eligible,
    DIRS["artifacts"]
    / "eligible_mouth_metadata.csv",
)


data_accounting = {
    "run_id": RUN_ID,
    "metadata_path": str(
        METADATA_PATH
    ),
    "total_metadata_rows": int(
        len(metadata)
    ),
    "accepted_status_rows": int(
        len(accepted)
    ),
    "audit_only_rows": int(
        len(audit_only)
    ),
    "missing_files_among_accepted": int(
        len(missing_file_rows)
    ),
    "training_eligible_success_count": int(
        len(eligible)
    ),
    "status_counts": {
        str(key): int(value)
        for key, value
        in status_counts.items()
    },
    "split_class_counts": {
        split_name: {
            label_name: int(
                (
                    (
                        eligible["split"]
                        == split_name
                    )
                    & (
                        eligible["label"]
                        == label_name
                    )
                ).sum()
            )
            for label_name in [
                "fake",
                "real",
            ]
        }
        for split_name in [
            "train",
            "val",
            "test",
        ]
    },
    "cross_split_path_intersections": (
        path_intersections
    ),
    "metadata_video_id_intersections": (
        video_intersections
    ),
    "video_leakage_status": (
        video_leakage_status
    ),
}


atomic_json_dump(
    data_accounting,
    RUN_DIR
    / "data_accounting.json",
)


print("=" * 80)
print("MOUTH METADATA VALIDATION PASSED")
print("=" * 80)

print("\nStatus counts:")
print(status_counts)

print("\nSplit/class counts:")
print(split_class_counts)

print("\nTotal eligible mouth ROI images:")
print(len(eligible))

print("\nCross-split path intersections:")
print(path_intersections)

print("\nMetadata video ID intersections:")
print(video_intersections)

print("\nVideo leakage status:")
print(video_leakage_status)

MOUTH METADATA VALIDATION PASSED

Status counts:
status
success    2987
skipped     111
Name: count, dtype: Int64

Split/class counts:
label  fake  real
split            
test    156   146
train  1192  1197
val     141   155

Total eligible mouth ROI images:
2987

Cross-split path intersections:
{'train_val': 0, 'train_test': 0, 'val_test': 0}

Metadata video ID intersections:
{'train_val': 0, 'train_test': 0, 'val_test': 0}

Video leakage status:
ZERO_INTERSECTION_FOR_METADATA_VIDEO_ID


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [15]:
# ============================================================
# CELL 9 — LOCAL SSD MOUTH IMAGE CACHE
# ============================================================

LOCAL_CACHE_ROOT = Path(
    "/content/mouth_roi_cache"
)

LOCAL_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def verify_image_file(
    path: Path,
) -> None:
    """
    Verify that an image file can be opened by PIL.
    """
    with Image.open(path) as image:
        image.verify()


def cache_one_image(
    source: Path,
    sample_id: str,
) -> Path:
    """
    Copy one mouth ROI image to the local Colab SSD.
    Existing valid cached images are reused.
    """
    suffix = (
        source.suffix.lower()
        if source.suffix
        else ".png"
    )

    destination = (
        LOCAL_CACHE_ROOT
        / f"{sample_id}{suffix}"
    )

    # Reuse an existing valid cache file.
    if destination.is_file():
        try:
            if destination.stat().st_size > 0:
                if CONFIG.verify_cached_images:
                    verify_image_file(
                        destination
                    )

                return destination

        except Exception:
            try:
                destination.unlink()
            except FileNotFoundError:
                pass

    # Copy to a temporary file first.
    temporary_path = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    shutil.copyfile(
        source,
        temporary_path,
    )

    if temporary_path.stat().st_size <= 0:
        raise RuntimeError(
            f"Copied cache file is empty: {source}"
        )

    if CONFIG.verify_cached_images:
        verify_image_file(
            temporary_path
        )

    os.replace(
        temporary_path,
        destination,
    )

    return destination


if CONFIG.cache_images_locally:
    cached_paths = []

    iterator = tqdm(
        eligible.itertuples(
            index=False
        ),
        total=len(eligible),
        desc=(
            "Caching mouth ROI images "
            "to local Colab SSD"
        ),
    )

    for row in iterator:
        source = Path(
            row.resolved_mouth_path
        )

        cached_path = cache_one_image(
            source=source,
            sample_id=str(
                row.sample_id
            ),
        )

        cached_paths.append(
            str(cached_path)
        )

    eligible[
        "training_mouth_path"
    ] = cached_paths

else:
    eligible[
        "training_mouth_path"
    ] = (
        eligible[
            "resolved_mouth_path"
        ].astype(str)
    )


# ------------------------------------------------------------
# Final cache validation
# ------------------------------------------------------------

if eligible[
    "training_mouth_path"
].isna().any():
    raise RuntimeError(
        "Missing training_mouth_path "
        "after the cache step."
    )


missing_cached_files = [
    path
    for path in eligible[
        "training_mouth_path"
    ]
    if not Path(path).is_file()
]


if missing_cached_files:
    raise RuntimeError(
        "Local cache validation failed. "
        f"Missing files: {len(missing_cached_files)}"
    )


atomic_csv_dump(
    eligible,
    DIRS["artifacts"]
    / "eligible_mouth_metadata.csv",
)


print("=" * 80)
print("LOCAL MOUTH CACHE PASSED")
print("=" * 80)

print(
    "Cached/usable mouth images:",
    len(eligible),
)

print(
    "Local cache root:",
    LOCAL_CACHE_ROOT,
)

Caching mouth ROI images to local Colab SSD:   0%|          | 0/2987 [00:00<?, ?it/s]

LOCAL MOUTH CACHE PASSED
Cached/usable mouth images: 2987
Local cache root: /content/mouth_roi_cache


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [16]:
# ============================================================
# CELL 10 — TRANSFORMS + DATASETS + DATALOADERS
# ============================================================

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225,
]


# ------------------------------------------------------------
# Training preprocessing and augmentation
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize(
        (
            CONFIG.image_size,
            CONFIG.image_size,
        ),
        interpolation=(
            transforms
            .InterpolationMode
            .BICUBIC
        ),
        antialias=True,
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD,
    ),
])


# ------------------------------------------------------------
# Deterministic validation and test preprocessing
# ------------------------------------------------------------

eval_transform = transforms.Compose([
    transforms.Resize(
        (
            CONFIG.image_size,
            CONFIG.image_size,
        ),
        interpolation=(
            transforms
            .InterpolationMode
            .BICUBIC
        ),
        antialias=True,
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD,
    ),
])


LABEL_TO_INT = {
    CONFIG.negative_label: 0,
    CONFIG.positive_label: 1,
}

INT_TO_LABEL = {
    0: CONFIG.negative_label,
    1: CONFIG.positive_label,
}


class MouthROIDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        transform,
    ):
        self.df = (
            frame
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(
        self,
        index,
    ):
        row = self.df.iloc[index]

        path = Path(
            row[
                "training_mouth_path"
            ]
        )

        try:
            with Image.open(path) as image:
                image = image.convert(
                    "RGB"
                )

                image_tensor = (
                    self.transform(
                        image
                    )
                )

        except Exception as error:
            raise RuntimeError(
                "Failed to read or transform "
                "mouth ROI image. "
                f"index={index}, path={path}"
            ) from error

        label_tensor = torch.tensor(
            LABEL_TO_INT[
                row["label"]
            ],
            dtype=torch.float32,
        )

        return {
            "image": image_tensor,
            "label": label_tensor,
            "sample_id": str(
                row["sample_id"]
            ),
            "path": str(path),
        }


# ------------------------------------------------------------
# Preserve the existing train / val / test split
# ------------------------------------------------------------

split_frames = {
    split_name: (
        eligible.loc[
            eligible["split"]
            == split_name
        ]
        .copy()
        .reset_index(drop=True)
    )
    for split_name in [
        "train",
        "val",
        "test",
    ]
}


datasets = {
    "train": MouthROIDataset(
        split_frames["train"],
        train_transform,
    ),

    "val": MouthROIDataset(
        split_frames["val"],
        eval_transform,
    ),

    "test": MouthROIDataset(
        split_frames["test"],
        eval_transform,
    ),
}


# Reproducible train shuffle generator
loader_generator = torch.Generator()

loader_generator.manual_seed(
    CONFIG.seed
)


loaders = {
    "train": DataLoader(
        datasets["train"],
        batch_size=CONFIG.batch_size,
        shuffle=True,
        num_workers=CONFIG.num_workers,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        generator=loader_generator,
        drop_last=False,
    ),

    "val": DataLoader(
        datasets["val"],
        batch_size=CONFIG.batch_size,
        shuffle=False,
        num_workers=CONFIG.num_workers,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        drop_last=False,
    ),

    "test": DataLoader(
        datasets["test"],
        batch_size=CONFIG.batch_size,
        shuffle=False,
        num_workers=CONFIG.num_workers,
        pin_memory=(
            DEVICE.type == "cuda"
        ),
        drop_last=False,
    ),
}


dataset_sizes = {
    split_name: len(loader.dataset)
    for split_name, loader
    in loaders.items()
}

print("Dataset sizes:", dataset_sizes)


# ------------------------------------------------------------
# One-batch shape and finite-value validation
# ------------------------------------------------------------

sample_batch = next(
    iter(
        loaders["train"]
    )
)

expected_shape = (
    3,
    CONFIG.image_size,
    CONFIG.image_size,
)

actual_shape = tuple(
    sample_batch[
        "image"
    ].shape[1:]
)


if actual_shape != expected_shape:
    raise RuntimeError(
        "Unexpected mouth ROI image shape: "
        f"{tuple(sample_batch['image'].shape)}"
    )


if not torch.isfinite(
    sample_batch["image"]
).all():
    raise FloatingPointError(
        "The sample batch contains NaN or Inf values."
    )


print("=" * 80)
print("TRANSFORMS + DATASET + DATALOADER PASSED")
print("=" * 80)

print(
    "Batch image shape:",
    tuple(
        sample_batch["image"].shape
    ),
)

print(
    "Batch label shape:",
    tuple(
        sample_batch["label"].shape
    ),
)

print(
    "Preprocessing:",
    "224x224 RGB + ImageNet normalization",
)

Dataset sizes: {'train': 2389, 'val': 296, 'test': 302}
TRANSFORMS + DATASET + DATALOADER PASSED
Batch image shape: (32, 3, 224, 224)
Batch label shape: (32,)
Preprocessing: 224x224 RGB + ImageNet normalization


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [17]:
# ============================================================
# CELL 11 — 5-BATCH I/O SPEED SANITY CHECK
# ============================================================

speed_test_batches = 5

start_time = time.time()
batches_seen = 0
images_seen = 0


for batch in tqdm(
    loaders["train"],
    total=min(
        speed_test_batches,
        len(loaders["train"]),
    ),
    desc="Mouth DataLoader speed test",
):
    images = batch["image"]

    # Validate the loaded tensors.
    if not torch.isfinite(
        images
    ).all():
        raise FloatingPointError(
            "NaN or Inf was found during "
            "the DataLoader speed test."
        )

    batches_seen += 1
    images_seen += len(images)

    if (
        batches_seen
        >= speed_test_batches
    ):
        break


elapsed_seconds = (
    time.time()
    - start_time
)


print("=" * 80)
print("DATALOADER SPEED TEST PASSED")
print("=" * 80)

print(
    f"{batches_seen} batches / "
    f"{images_seen} mouth images loaded "
    f"in {elapsed_seconds:.2f} seconds."
)


if elapsed_seconds > 60:
    print(
        "WARNING: Local data loading is unusually slow. "
        "Training can still run, but Colab disk/runtime "
        "performance should be monitored."
    )
else:
    print(
        "DataLoader speed looks healthy."
    )

Mouth DataLoader speed test:   0%|          | 0/5 [00:00<?, ?it/s]

DATALOADER SPEED TEST PASSED
5 batches / 160 mouth images loaded in 0.86 seconds.
DataLoader speed looks healthy.


In [18]:
# ============================================================
# CELL 12 — EFFICIENTNET-B0 MODEL
# ============================================================

def build_model(
    pretrained: bool = True,
) -> nn.Module:
    """
    Build EfficientNet-B0 for binary real/fake
    mouth ROI classification.
    """
    weights = (
        EfficientNet_B0_Weights.DEFAULT
        if pretrained
        else None
    )

    try:
        model_instance = efficientnet_b0(
            weights=weights
        )

    except Exception as error:
        raise RuntimeError(
            "EfficientNet-B0 pretrained weights "
            "could not be loaded. Check the Colab "
            "internet connection and runtime."
        ) from error

    classifier_input_features = (
        model_instance
        .classifier[1]
        .in_features
    )

    # Replace the original 1000-class ImageNet head
    # with a one-logit binary classification head.
    model_instance.classifier = nn.Sequential(
        nn.Dropout(
            p=CONFIG.dropout
        ),
        nn.Linear(
            classifier_input_features,
            1,
        ),
    )

    return model_instance


model = build_model(
    pretrained=True
).to(DEVICE)


total_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)


# Basic forward-pass shape test
test_images = (
    sample_batch["image"][:2]
    .to(DEVICE)
)

model.eval()

with torch.no_grad():
    test_logits = (
        model(test_images)
        .squeeze(1)
    )


if tuple(
    test_logits.shape
) != (len(test_images),):
    raise RuntimeError(
        "Unexpected EfficientNet-B0 output shape: "
        f"{tuple(test_logits.shape)}"
    )


if not torch.isfinite(
    test_logits
).all():
    raise FloatingPointError(
        "EfficientNet-B0 produced NaN or Inf logits."
    )


print("=" * 80)
print("EFFICIENTNET-B0 MODEL PASSED")
print("=" * 80)

print(
    "Model:",
    "torchvision EfficientNet-B0",
)

print(
    "Pretrained weights:",
    "EfficientNet_B0_Weights.DEFAULT",
)

print(
    "Input:",
    f"RGB mouth ROI "
    f"{CONFIG.image_size}x{CONFIG.image_size}",
)

print(
    "Output:",
    "One binary logit for P(fake)",
)

print(
    "Total parameters:",
    f"{total_parameters:,}",
)

print(
    "Currently trainable parameters:",
    f"{trainable_parameters:,}",
)

print(
    "Test output shape:",
    tuple(test_logits.shape),
)

print(
    "Device:",
    DEVICE,
)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 92.6MB/s]


EFFICIENTNET-B0 MODEL PASSED
Model: torchvision EfficientNet-B0
Pretrained weights: EfficientNet_B0_Weights.DEFAULT
Input: RGB mouth ROI 224x224
Output: One binary logit for P(fake)
Total parameters: 4,008,829
Currently trainable parameters: 4,008,829
Test output shape: (2,)
Device: cpu


In [19]:
# ============================================================
# CELL 13 — METRICS
# ============================================================

def safe_roc_auc(y_true, probabilities):
    if len(np.unique(y_true)) < 2:
        return float("nan")

    return float(roc_auc_score(y_true, probabilities))


def safe_average_precision(y_true, probabilities):
    if len(np.unique(y_true)) < 2:
        return float("nan")

    return float(
        average_precision_score(y_true, probabilities)
    )


def binary_metrics(
    y_true,
    probabilities,
    threshold: float,
) -> Dict[str, float]:

    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else float("nan")
    )

    return {
        "threshold": float(threshold),
        "accuracy": float(
            accuracy_score(y_true, predictions)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, predictions)
        ),
        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "specificity": float(specificity),
        "roc_auc": safe_roc_auc(
            y_true,
            probabilities,
        ),
        "average_precision": safe_average_precision(
            y_true,
            probabilities,
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def select_validation_threshold(
    y_true,
    probabilities,
):
    thresholds = np.linspace(
        CONFIG.threshold_min,
        CONFIG.threshold_max,
        CONFIG.threshold_steps,
    )

    rows = [
        binary_metrics(
            y_true,
            probabilities,
            float(threshold),
        )
        for threshold in thresholds
    ]

    table = pd.DataFrame(rows)

    table["distance_to_0_5"] = (
        table["threshold"] - 0.5
    ).abs()

    best = (
        table
        .sort_values(
            [
                "f1",
                "balanced_accuracy",
                "distance_to_0_5",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .iloc[0]
    )

    return (
        float(best["threshold"]),
        table.drop(columns=["distance_to_0_5"]),
    )


print("=" * 72)
print("METRICS FUNCTIONS PASSED")
print("=" * 72)
print("Binary metric calculation      : READY")
print("Validation threshold selection : READY")

METRICS FUNCTIONS PASSED
Binary metric calculation      : READY
Validation threshold selection : READY


In [20]:
# ============================================================
# CELL 14 — SAFE CHECKPOINT + RNG HELPERS
# ============================================================

CHECKPOINT_SCHEMA_VERSION = 2


def get_rng_state() -> dict:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state().cpu(),
        "loader_generator": loader_generator.get_state().cpu(),
    }

    if torch.cuda.is_available():
        state["cuda"] = [
            cuda_state.cpu()
            for cuda_state in torch.cuda.get_rng_state_all()
        ]

    return state


def _to_cpu_byte_tensor(value) -> torch.Tensor:
    if not torch.is_tensor(value):
        value = torch.tensor(
            value,
            dtype=torch.uint8,
        )

    return (
        value
        .detach()
        .cpu()
        .to(dtype=torch.uint8)
    )


def set_rng_state(
    state: Optional[dict],
) -> None:

    if not state:
        return

    python_state = state.get("python")

    if python_state is not None:
        random.setstate(python_state)

    numpy_state = state.get("numpy")

    if numpy_state is not None:
        np.random.set_state(numpy_state)

    torch_state = state.get("torch")

    if torch_state is not None:
        torch.set_rng_state(
            _to_cpu_byte_tensor(torch_state)
        )

    loader_state = state.get("loader_generator")

    if loader_state is not None:
        loader_generator.set_state(
            _to_cpu_byte_tensor(loader_state)
        )

    cuda_states = state.get("cuda")

    if (
        torch.cuda.is_available()
        and cuda_states is not None
    ):
        safe_states = [
            _to_cpu_byte_tensor(cuda_state)
            for cuda_state in cuda_states
        ]

        if len(safe_states) == torch.cuda.device_count():
            torch.cuda.set_rng_state_all(safe_states)
        else:
            print(
                "WARNING: CUDA RNG device count differs; "
                "CUDA RNG restore skipped."
            )


def save_checkpoint(
    path: Path,
    *,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    best_score: float,
    history: List[dict],
    stage_name: str,
) -> None:

    state = {
        "checkpoint_schema_version": CHECKPOINT_SCHEMA_VERSION,
        "run_schema_version": RUN_SCHEMA_VERSION,
        "epoch": int(epoch),
        "stage_name": stage_name,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),

        "scheduler_state_dict": (
            scheduler.state_dict()
            if scheduler is not None
            else None
        ),

        "best_metric_score": float(best_score),
        "history": history,
        "config": asdict(CONFIG),
        "rng_state": get_rng_state(),
    }

    atomic_torch_save(state, path)


def checkpoint_is_compatible(
    path: Path,
    stage_name: str,
) -> bool:

    if not path.is_file():
        return False

    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

        return (
            int(
                checkpoint.get(
                    "checkpoint_schema_version",
                    -1,
                )
            )
            == CHECKPOINT_SCHEMA_VERSION

            and int(
                checkpoint.get(
                    "run_schema_version",
                    -1,
                )
            )
            == RUN_SCHEMA_VERSION

            and checkpoint.get("stage_name")
            == stage_name
        )

    except Exception:
        return False


def load_checkpoint(
    path: Path,
    *,
    stage_name: str,
    model: nn.Module,
    optimizer: Optional[
        torch.optim.Optimizer
    ] = None,
    scheduler=None,
) -> dict:

    if not checkpoint_is_compatible(
        path,
        stage_name,
    ):
        raise RuntimeError(
            "Checkpoint is missing, corrupt, or incompatible:\n"
            f"{path}"
        )

    # Checkpoint önce güvenli şekilde CPU'ya yüklenir.
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.to(DEVICE)

    if optimizer is not None:
        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        # Optimizer tensörlerini mevcut cihaza taşı.
        for optimizer_state in optimizer.state.values():
            for key, value in list(
                optimizer_state.items()
            ):
                if torch.is_tensor(value):
                    optimizer_state[key] = value.to(DEVICE)

    if (
        scheduler is not None
        and checkpoint.get(
            "scheduler_state_dict"
        ) is not None
    ):
        scheduler.load_state_dict(
            checkpoint["scheduler_state_dict"]
        )

    set_rng_state(
        checkpoint.get("rng_state")
    )

    return checkpoint


print("=" * 72)
print("SAFE CHECKPOINT + RNG HELPERS PASSED")
print("=" * 72)
print("Checkpoint schema version :", CHECKPOINT_SCHEMA_VERSION)
print("Checkpoint save/load      : READY")
print("RNG state save/restore     : READY")

SAFE CHECKPOINT + RNG HELPERS PASSED
Checkpoint schema version : 2
Checkpoint save/load      : READY
RNG state save/restore     : READY


In [21]:
# ============================================================
# CELL 15 — SAFE FP32 EPOCH RUNNER WITH LIVE PROGRESS
# ============================================================

criterion = nn.BCEWithLogitsLoss()


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    *,
    training: bool,
    optimizer: Optional[
        torch.optim.Optimizer
    ] = None,
    frozen_backbone: bool = False,
    description: str = "",
) -> dict:

    if training:
        if optimizer is None:
            raise ValueError(
                "optimizer is required when training=True"
            )

        model.train()

        # Omurga dondurulduğunda feature katmanlarındaki
        # BatchNorm ve dropout da dondurulur.
        if frozen_backbone:
            model.features.eval()
            model.classifier.train()

    else:
        model.eval()

    total_loss = 0.0
    processed = 0

    all_labels = []
    all_probabilities = []
    all_sample_ids = []
    all_paths = []

    progress = tqdm(
        loader,
        total=len(loader),
        desc=description,
        leave=False,
    )

    for batch_index, batch in enumerate(progress):
        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )

        labels = batch["label"].to(
            DEVICE,
            non_blocking=True,
        ).float()

        if not torch.isfinite(images).all():
            raise FloatingPointError(
                f"NaN/Inf detected in images at batch={batch_index}"
            )

        if not torch.isfinite(labels).all():
            raise FloatingPointError(
                f"NaN/Inf detected in labels at batch={batch_index}"
            )

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            # Sayısal kararlılık için bilinçli şekilde FP32.
            logits = (
                model(images)
                .squeeze(1)
                .float()
            )

            if not torch.isfinite(logits).all():
                raise FloatingPointError(
                    f"Non-finite logits at batch={batch_index}"
                )

            loss = criterion(
                logits,
                labels,
            )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite loss at batch={batch_index}"
                )

            if training:
                loss.backward()

                bad_gradient_names = []

                for (
                    parameter_name,
                    parameter,
                ) in model.named_parameters():

                    if parameter.grad is None:
                        continue

                    if not torch.isfinite(
                        parameter.grad
                    ).all():
                        bad_gradient_names.append(
                            parameter_name
                        )

                if bad_gradient_names:
                    raise FloatingPointError(
                        "Non-finite gradients detected at "
                        f"batch={batch_index}. "
                        "First affected parameters: "
                        f"{bad_gradient_names[:10]}"
                    )

                gradient_norm = (
                    torch.nn.utils
                    .clip_grad_norm_(
                        model.parameters(),
                        CONFIG.grad_clip_norm,
                        error_if_nonfinite=True,
                    )
                )

                if not torch.isfinite(
                    torch.as_tensor(gradient_norm)
                ):
                    raise FloatingPointError(
                        f"Non-finite gradient norm at batch={batch_index}"
                    )

                optimizer.step()

        probabilities = torch.sigmoid(
            logits.detach()
        )

        if not torch.isfinite(probabilities).all():
            raise FloatingPointError(
                f"Non-finite probabilities at batch={batch_index}"
            )

        batch_size = images.size(0)

        total_loss += (
            float(loss.detach().item())
            * batch_size
        )

        processed += batch_size

        all_labels.extend(
            labels
            .detach()
            .cpu()
            .numpy()
            .astype(int)
            .tolist()
        )

        all_probabilities.extend(
            probabilities
            .detach()
            .cpu()
            .numpy()
            .astype(float)
            .tolist()
        )

        all_sample_ids.extend(
            list(batch["sample_id"])
        )

        all_paths.extend(
            list(batch["path"])
        )

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            processed=processed,
        )

    if processed != len(loader.dataset):
        raise RuntimeError(
            "Epoch accounting mismatch: "
            f"processed={processed}, "
            f"dataset={len(loader.dataset)}"
        )

    return {
        "loss": (
            total_loss
            / max(processed, 1)
        ),
        "labels": np.asarray(
            all_labels,
            dtype=int,
        ),
        "probabilities": np.asarray(
            all_probabilities,
            dtype=float,
        ),
        "sample_ids": all_sample_ids,
        "paths": all_paths,
        "count": int(processed),
    }


print("=" * 72)
print("SAFE FP32 EPOCH RUNNER PASSED")
print("=" * 72)
print("Loss function       : BCEWithLogitsLoss")
print("Numerical precision : FP32")
print("Gradient clipping   :", CONFIG.grad_clip_norm)
print("Epoch runner        : READY")

SAFE FP32 EPOCH RUNNER PASSED
Loss function       : BCEWithLogitsLoss
Numerical precision : FP32
Gradient clipping   : 1.0
Epoch runner        : READY


In [22]:
# ============================================================
# CELL 16 — TWO-BATCH MODEL SMOKE TEST
# ============================================================

def model_smoke_test(
    model: nn.Module,
) -> None:

    print(
        "Running two-batch forward/backward smoke test..."
    )

    # Testten sonra modeli eski durumuna döndürmek için
    # mevcut ağırlıkları yedekle.
    backup_state = {
        key: value
        .detach()
        .cpu()
        .clone()
        for key, value in model.state_dict().items()
    }

    temporary_optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-6,
    )

    model.train()
    seen_batches = 0

    for batch in tqdm(
        loaders["train"],
        total=min(
            2,
            len(loaders["train"]),
        ),
        desc="Smoke test",
        leave=False,
    ):
        images = batch["image"].to(DEVICE)

        labels = (
            batch["label"]
            .to(DEVICE)
            .float()
        )

        temporary_optimizer.zero_grad(
            set_to_none=True
        )

        logits = (
            model(images)
            .squeeze(1)
            .float()
        )

        loss = criterion(
            logits,
            labels,
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                "Smoke test produced non-finite loss."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            CONFIG.grad_clip_norm,
            error_if_nonfinite=True,
        )

        temporary_optimizer.step()

        seen_batches += 1

        if seen_batches >= 2:
            break

    if seen_batches < 2:
        raise RuntimeError(
            "Smoke test could not obtain two batches."
        )

    # Smoke test sırasında oluşan geçici değişiklikleri geri al.
    model.load_state_dict(backup_state)
    model.to(DEVICE)

    del backup_state, temporary_optimizer

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("=" * 72)
    print("TWO-BATCH MODEL SMOKE TEST PASSED")
    print("=" * 72)
    print("Processed batches : 2")
    print("Forward pass      : PASSED")
    print("Backward pass     : PASSED")
    print("Gradient check    : PASSED")


model_smoke_test(model)

Running two-batch forward/backward smoke test...


Smoke test:   0%|          | 0/2 [00:00<?, ?it/s]

NameError: name 'gc' is not defined

In [23]:
import gc

model_smoke_test(model)

Running two-batch forward/backward smoke test...


Smoke test:   0%|          | 0/2 [00:00<?, ?it/s]

TWO-BATCH MODEL SMOKE TEST PASSED
Processed batches : 2
Forward pass      : PASSED
Backward pass     : PASSED
Gradient check    : PASSED


In [24]:
# ============================================================
# CELL 17 — TRAINING FUNCTION
# ============================================================

def train_stage(
    *,
    model: nn.Module,
    stage_name: str,
    epochs: int,
    lr: float,
    stage_dir: Path,
    frozen_backbone: bool,
    allow_resume: bool,
) -> dict:

    trainable_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    if not trainable_parameters:
        raise RuntimeError(
            f"No trainable parameters for stage={stage_name}"
        )

    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=lr,
        weight_decay=CONFIG.weight_decay,
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=CONFIG.scheduler_patience,
            min_lr=1e-7,
        )
    )

    last_checkpoint = stage_dir / "last.ckpt"
    best_checkpoint = stage_dir / "best.ckpt"

    history = []
    start_epoch = 1
    best_score = -float("inf")
    no_improvement_epochs = 0

    if (
        allow_resume
        and checkpoint_is_compatible(
            last_checkpoint,
            stage_name,
        )
    ):
        print(
            f"[{stage_name}] Resuming compatible checkpoint:"
        )
        print(last_checkpoint)

        checkpoint = load_checkpoint(
            last_checkpoint,
            stage_name=stage_name,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
        )

        history = list(
            checkpoint.get("history", [])
        )

        start_epoch = int(checkpoint["epoch"]) + 1

        best_score = float(
            checkpoint.get(
                "best_metric_score",
                -float("inf"),
            )
        )

        if history:
            monitor_scores = [
                row.get(
                    "val_roc_auc",
                    float("nan"),
                )
                for row in history
            ]

            safe_scores = [
                score
                if np.isfinite(score)
                else -float("inf")
                for score in monitor_scores
            ]

            best_index = int(
                np.argmax(safe_scores)
            )

            no_improvement_epochs = max(
                0,
                len(history) - best_index - 1,
            )

    if start_epoch > epochs:
        print(
            f"[{stage_name}] Stage already complete."
        )

        if not checkpoint_is_compatible(
            best_checkpoint,
            stage_name,
        ):
            raise RuntimeError(
                "Stage says complete but compatible "
                "best.ckpt is missing: "
                f"{best_checkpoint}"
            )

        return {
            "stage": stage_name,
            "history": history,
            "best_score": best_score,
            "best_checkpoint": str(best_checkpoint),
            "last_checkpoint": str(last_checkpoint),
        }

    for epoch in range(
        start_epoch,
        epochs + 1,
    ):
        epoch_start = time.time()

        train_output = run_epoch(
            model,
            loaders["train"],
            training=True,
            optimizer=optimizer,
            frozen_backbone=frozen_backbone,
            description=(
                f"{stage_name} train {epoch}/{epochs}"
            ),
        )

        validation_output = run_epoch(
            model,
            loaders["val"],
            training=False,
            frozen_backbone=False,
            description=(
                f"{stage_name} val {epoch}/{epochs}"
            ),
        )

        train_metrics = binary_metrics(
            train_output["labels"],
            train_output["probabilities"],
            threshold=0.5,
        )

        validation_metrics = binary_metrics(
            validation_output["labels"],
            validation_output["probabilities"],
            threshold=0.5,
        )

        monitor = validation_metrics["roc_auc"]

        if not np.isfinite(monitor):
            monitor = validation_metrics["f1"]

        scheduler.step(monitor)

        row = {
            "stage": stage_name,
            "epoch": int(epoch),

            "lr": float(
                optimizer.param_groups[0]["lr"]
            ),

            "train_loss": float(
                train_output["loss"]
            ),

            "val_loss": float(
                validation_output["loss"]
            ),

            **{
                f"train_{key}": value
                for key, value
                in train_metrics.items()
            },

            **{
                f"val_{key}": value
                for key, value
                in validation_metrics.items()
            },

            "epoch_seconds": float(
                time.time() - epoch_start
            ),
        }

        history.append(row)

        improved = (
            monitor > best_score + 1e-8
        )

        if improved:
            best_score = float(monitor)
            no_improvement_epochs = 0
        else:
            no_improvement_epochs += 1

        # Tamamlanan her epoch kaldığı yerden
        # devam edilebilmesi için kaydedilir.
        save_checkpoint(
            last_checkpoint,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            best_score=best_score,
            history=history,
            stage_name=stage_name,
        )

        if improved:
            save_checkpoint(
                best_checkpoint,
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                best_score=best_score,
                history=history,
                stage_name=stage_name,
            )

        atomic_csv_dump(
            pd.DataFrame(history),
            DIRS["metrics"]
            / f"{stage_name}_training_history.csv",
        )

        print(
            f"[{stage_name}] epoch {epoch:02d}/{epochs} | "
            f"train_loss={row['train_loss']:.5f} | "
            f"val_loss={row['val_loss']:.5f} | "
            f"val_auc={row['val_roc_auc']:.5f} | "
            f"val_f1={row['val_f1']:.5f} | "
            f"lr={row['lr']:.2e} | "
            f"time={row['epoch_seconds']:.1f}s"
        )

        if (
            no_improvement_epochs
            >= CONFIG.early_stopping_patience
        ):
            print(
                f"[{stage_name}] Early stopping after "
                f"{no_improvement_epochs} "
                "non-improving epoch(s)."
            )
            break

    if not checkpoint_is_compatible(
        best_checkpoint,
        stage_name,
    ):
        raise RuntimeError(
            "No compatible best checkpoint created "
            f"for stage={stage_name}"
        )

    return {
        "stage": stage_name,
        "history": history,
        "best_score": best_score,
        "best_checkpoint": str(best_checkpoint),
        "last_checkpoint": str(last_checkpoint),
    }


print("=" * 72)
print("TRAINING FUNCTION PASSED")
print("=" * 72)
print("Optimizer       : AdamW")
print("Scheduler       : ReduceLROnPlateau")
print("Early stopping  :", CONFIG.early_stopping_patience)
print("Checkpointing   : READY")
print("Resume support  : READY")

TRAINING FUNCTION PASSED
Optimizer       : AdamW
Scheduler       : ReduceLROnPlateau
Early stopping  : 4
Checkpointing   : READY
Resume support  : READY


In [25]:
# ============================================================
# CELL 18 — STAGE 1: FROZEN EFFICIENTNET-B0 BACKBONE
# ============================================================

# EfficientNet-B0 özellik çıkarım omurgasını dondur.
for parameter in model.features.parameters():
    parameter.requires_grad = False

# Yalnızca son sınıflandırıcı katmanını eğit.
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

frozen_trainable = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("=" * 72)
print("STAGE 1 — FROZEN BACKBONE TRAINING")
print("=" * 72)
print("Device                   :", DEVICE)
print("Trainable parameters     :", f"{frozen_trainable:,}")
print("Maximum epochs           :", CONFIG.frozen_epochs)
print("Learning rate            :", CONFIG.frozen_lr)
print("Resume/checkpoint support: ENABLED")

if DEVICE.type == "cpu":
    print(
        "NOTE: CPU is active. Training will be slow, "
        "but completed epochs will be saved."
    )

frozen_summary = train_stage(
    model=model,
    stage_name="frozen",
    epochs=CONFIG.frozen_epochs,
    lr=CONFIG.frozen_lr,
    stage_dir=FROZEN_DIR,
    frozen_backbone=True,
    allow_resume=True,
)

atomic_json_dump(
    frozen_summary,
    DIRS["metrics"]
    / "frozen_training_summary.json",
)

print("=" * 72)
print("STAGE 1 — FROZEN TRAINING COMPLETED")
print("=" * 72)
print("Best validation score :", frozen_summary["best_score"])
print("Best checkpoint       :", frozen_summary["best_checkpoint"])

frozen_summary

STAGE 1 — FROZEN BACKBONE TRAINING
Device                   : cpu
Trainable parameters     : 1,281
Maximum epochs           : 5
Learning rate            : 0.001
Resume/checkpoint support: ENABLED
NOTE: CPU is active. Training will be slow, but completed epochs will be saved.


frozen train 1/5:   0%|          | 0/75 [00:00<?, ?it/s]

frozen val 1/5:   0%|          | 0/10 [00:00<?, ?it/s]

[frozen] epoch 01/5 | train_loss=0.65702 | val_loss=0.62411 | val_auc=0.71229 | val_f1=0.63469 | lr=1.00e-03 | time=173.5s


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

frozen train 5/5:   0%|          | 0/75 [00:00<?, ?it/s]

frozen val 5/5:   0%|          | 0/10 [00:00<?, ?it/s]

[frozen] epoch 05/5 | train_loss=0.58380 | val_loss=0.58587 | val_auc=0.75484 | val_f1=0.65018 | lr=1.00e-03 | time=173.5s
STAGE 1 — FROZEN TRAINING COMPLETED
Best validation score : 0.7548387096774194
Best checkpoint       : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar/20260808_1257_mouth_efficientnet_b0_seed42/checkpoints/frozen/best.ckpt


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


{'stage': 'frozen',
 'history': [{'stage': 'frozen',
   'epoch': 1,
   'lr': 0.001,
   'train_loss': 0.6570216026102563,
   'val_loss': 0.6241069564948211,
   'train_threshold': 0.5,
   'train_accuracy': 0.594390958560067,
   'train_balanced_accuracy': 0.5944233486400565,
   'train_precision': 0.5905767668562144,
   'train_recall': 0.6098993288590604,
   'train_f1': 0.6000825423029302,
   'train_specificity': 0.5789473684210527,
   'train_roc_auc': 0.6444543966179431,
   'train_average_precision': 0.6533092614601864,
   'train_tn': 693,
   'train_fp': 504,
   'train_fn': 465,
   'train_tp': 727,
   'val_threshold': 0.5,
   'val_accuracy': 0.6655405405405406,
   'val_balanced_accuracy': 0.6630290551361244,
   'val_precision': 0.6615384615384615,
   'val_recall': 0.6099290780141844,
   'val_f1': 0.6346863468634686,
   'val_specificity': 0.7161290322580646,
   'val_roc_auc': 0.7122855181880576,
   'val_average_precision': 0.6889986274115011,
   'val_tn': 111,
   'val_fp': 44,
   'val_fn':

In [26]:
# ============================================================
# CELL 19 — STAGE 2: FULL FP32 FINE-TUNING
# ============================================================

FROZEN_BEST = FROZEN_DIR / "best.ckpt"

if not checkpoint_is_compatible(
    FROZEN_BEST,
    "frozen",
):
    raise RuntimeError(
        "Frozen best checkpoint is missing or incompatible."
    )

# Stage 2 her zaman Stage 1'in en iyi ağırlıklarıyla başlar.
frozen_checkpoint = torch.load(
    FROZEN_BEST,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    frozen_checkpoint["model_state_dict"]
)

model.to(DEVICE)

# Fine-tuning için modelin bütün katmanlarını aç.
for parameter in model.parameters():
    parameter.requires_grad = True

finetune_trainable = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("=" * 72)
print("STAGE 2 — FULL FP32 FINE-TUNING")
print("=" * 72)
print("Device                    :", DEVICE)
print("Trainable parameters      :", f"{finetune_trainable:,}")
print("Maximum epochs            :", CONFIG.finetune_epochs)
print("Learning rate             :", CONFIG.finetune_lr)
print("Initialization checkpoint :", FROZEN_BEST)
print("Resume/checkpoint support : ENABLED")

if DEVICE.type == "cpu":
    print(
        "NOTE: CPU is active. Full fine-tuning will be slow, "
        "but every completed epoch will be saved."
    )

# Aynı çalıştırmaya ait uyumlu checkpoint varsa
# tamamlanan son epoch'tan devam eder.
finetune_summary = train_stage(
    model=model,
    stage_name="finetune",
    epochs=CONFIG.finetune_epochs,
    lr=CONFIG.finetune_lr,
    stage_dir=FINETUNE_DIR,
    frozen_backbone=False,
    allow_resume=True,
)

atomic_json_dump(
    finetune_summary,
    DIRS["metrics"]
    / "finetune_training_summary.json",
)

print("=" * 72)
print("STAGE 2 — FINE-TUNING COMPLETED")
print("=" * 72)
print(
    "Best validation score :",
    finetune_summary["best_score"],
)
print(
    "Best checkpoint       :",
    finetune_summary["best_checkpoint"],
)

finetune_summary

STAGE 2 — FULL FP32 FINE-TUNING
Device                    : cpu
Trainable parameters      : 4,008,829
Maximum epochs            : 15
Learning rate             : 2e-05
Initialization checkpoint : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar/20260808_1257_mouth_efficientnet_b0_seed42/checkpoints/frozen/best.ckpt
Resume/checkpoint support : ENABLED
NOTE: CPU is active. Full fine-tuning will be slow, but every completed epoch will be saved.


finetune train 1/15:   0%|          | 0/75 [00:00<?, ?it/s]

finetune val 1/15:   0%|          | 0/10 [00:00<?, ?it/s]

[finetune] epoch 01/15 | train_loss=0.70605 | val_loss=0.66513 | val_auc=0.66548 | val_f1=0.54852 | lr=2.00e-05 | time=733.6s


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

finetune train 15/15:   0%|          | 0/75 [00:00<?, ?it/s]

finetune val 15/15:   0%|          | 0/10 [00:00<?, ?it/s]

[finetune] epoch 15/15 | train_loss=0.33953 | val_loss=0.61501 | val_auc=0.78929 | val_f1=0.65882 | lr=2.00e-05 | time=712.3s
STAGE 2 — FINE-TUNING COMPLETED
Best validation score : 0.7968885838480896
Best checkpoint       : /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar/20260808_1257_mouth_efficientnet_b0_seed42/checkpoints/finetune/best.ckpt


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


{'stage': 'finetune',
 'history': [{'stage': 'finetune',
   'epoch': 1,
   'lr': 2e-05,
   'train_loss': 0.7060461148101947,
   'val_loss': 0.6651304348095043,
   'train_threshold': 0.5,
   'train_accuracy': 0.5717873587275011,
   'train_balanced_accuracy': 0.5712081518112956,
   'train_precision': 0.6585365853658537,
   'train_recall': 0.29446308724832215,
   'train_f1': 0.40695652173913044,
   'train_specificity': 0.847953216374269,
   'train_roc_auc': 0.6378747483922333,
   'train_average_precision': 0.6276596572858624,
   'train_tn': 1015,
   'train_fp': 182,
   'train_fn': 841,
   'train_tp': 351,
   'val_threshold': 0.5,
   'val_accuracy': 0.6385135135135135,
   'val_balanced_accuracy': 0.6304964539007092,
   'val_precision': 0.6770833333333334,
   'val_recall': 0.46099290780141844,
   'val_f1': 0.5485232067510548,
   'val_specificity': 0.8,
   'val_roc_auc': 0.6654770075497597,
   'val_average_precision': 0.6463794803674635,
   'val_tn': 124,
   'val_fp': 31,
   'val_fn': 76,
  

In [27]:
# ============================================================
# CELL 20 — VALIDATION THRESHOLD SELECTION
# ============================================================

FINAL_BEST_CHECKPOINT = FINETUNE_DIR / "best.ckpt"

if not checkpoint_is_compatible(
    FINAL_BEST_CHECKPOINT,
    "finetune",
):
    raise RuntimeError(
        "Final best fine-tune checkpoint is missing "
        "or incompatible."
    )

# En iyi fine-tuning checkpoint'ini yükle.
final_checkpoint = torch.load(
    FINAL_BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    final_checkpoint["model_state_dict"]
)

model.to(DEVICE)
model.eval()

# Validation tahminlerini üret.
validation_output = run_epoch(
    model,
    loaders["val"],
    training=False,
    description="Final validation",
)

# En iyi eşik yalnızca validation verisinden seçilir.
best_threshold, threshold_table = (
    select_validation_threshold(
        validation_output["labels"],
        validation_output["probabilities"],
    )
)

validation_selected_metrics = binary_metrics(
    validation_output["labels"],
    validation_output["probabilities"],
    threshold=best_threshold,
)

# Eşik arama tablosunu ve validation metriklerini kaydet.
atomic_csv_dump(
    threshold_table,
    DIRS["metrics"]
    / "validation_threshold_search.csv",
)

atomic_json_dump(
    validation_selected_metrics,
    DIRS["metrics"]
    / "validation_selected_threshold_metrics.json",
)

print("=" * 72)
print("VALIDATION THRESHOLD SELECTION COMPLETED")
print("=" * 72)
print("Selected threshold :", best_threshold)
print(
    json.dumps(
        validation_selected_metrics,
        indent=2,
    )
)

Final validation:   0%|          | 0/10 [00:00<?, ?it/s]

VALIDATION THRESHOLD SELECTION COMPLETED
Selected threshold : 0.19
{
  "threshold": 0.19,
  "accuracy": 0.6993243243243243,
  "balanced_accuracy": 0.7080988332189431,
  "precision": 0.63,
  "recall": 0.8936170212765957,
  "f1": 0.7390029325513197,
  "specificity": 0.5225806451612903,
  "roc_auc": 0.7968885838480896,
  "average_precision": 0.7783525916777316,
  "tn": 81,
  "fp": 74,
  "fn": 15,
  "tp": 126
}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [28]:
# ============================================================
# CELL 21 — FINAL TEST EVALUATION
# ============================================================

test_output = run_epoch(
    model,
    loaders["test"],
    training=False,
    description="Final test",
)

test_metrics = binary_metrics(
    test_output["labels"],
    test_output["probabilities"],
    threshold=best_threshold,
)

test_predictions = pd.DataFrame({
    "sample_id": test_output["sample_ids"],
    "path": test_output["paths"],
    "label_int": test_output["labels"],

    "label": [
        INT_TO_LABEL[int(value)]
        for value in test_output["labels"]
    ],

    "prob_fake": test_output["probabilities"],
})

test_predictions["threshold"] = best_threshold

test_predictions["pred_int"] = (
    test_predictions["prob_fake"].to_numpy()
    >= best_threshold
).astype(int)

test_predictions["prediction"] = (
    test_predictions["pred_int"]
    .map(INT_TO_LABEL)
)

test_predictions["correct"] = (
    test_predictions["label_int"]
    == test_predictions["pred_int"]
)

# Final test metriklerini kaydet.
atomic_csv_dump(
    pd.DataFrame([
        {
            "model": "EfficientNet-B0",
            "roi_region": "mouth",
            **test_metrics,
        }
    ]),
    DIRS["metrics"]
    / "final_test_metrics.csv",
)

# Her test görüntüsünün tahminini kaydet.
atomic_csv_dump(
    test_predictions,
    DIRS["predictions"]
    / "test_predictions.csv",
)

print("=" * 72)
print("FINAL TEST EVALUATION COMPLETED")
print("=" * 72)
print("Test sample count :", len(test_predictions))
print("Applied threshold :", best_threshold)
print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)

Final test:   0%|          | 0/10 [00:00<?, ?it/s]

FINAL TEST EVALUATION COMPLETED
Test sample count : 302
Applied threshold : 0.19
{
  "threshold": 0.19,
  "accuracy": 0.7152317880794702,
  "balanced_accuracy": 0.7085528626624518,
  "precision": 0.6635514018691588,
  "recall": 0.9102564102564102,
  "f1": 0.7675675675675676,
  "specificity": 0.5068493150684932,
  "roc_auc": 0.8331577098700387,
  "average_precision": 0.8527717483914827,
  "tn": 74,
  "fp": 72,
  "fn": 14,
  "tp": 142
}


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [29]:
# ============================================================
# CELL 22 — 600 DPI FIGURES
# ============================================================

def save_figure(
    figure,
    filename: str,
) -> Path:

    path = DIRS["figures"] / filename

    figure.savefig(
        path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(figure)

    with Image.open(path) as image:
        if min(image.size) < 600:
            raise RuntimeError(
                "Figure resolution quality gate failed: "
                f"{path} -> {image.size}"
            )

    return path


frozen_history = pd.read_csv(
    DIRS["metrics"]
    / "frozen_training_history.csv"
)

finetune_history = pd.read_csv(
    DIRS["metrics"]
    / "finetune_training_history.csv"
)

history = pd.concat(
    [
        frozen_history,
        finetune_history,
    ],
    ignore_index=True,
)

history["global_epoch"] = np.arange(
    1,
    len(history) + 1,
)

saved_figures = []


# ------------------------------------------------------------
# 1. Training and validation loss
# ------------------------------------------------------------

figure, axis = plt.subplots(
    figsize=(10, 6)
)

axis.plot(
    history["global_epoch"],
    history["train_loss"],
    color="#1f77b4",
    linewidth=2,
    label="Training Loss",
)

axis.plot(
    history["global_epoch"],
    history["val_loss"],
    color="#ff7f0e",
    linewidth=2,
    label="Validation Loss",
)

axis.set_title(
    "EfficientNet-B0 Mouth ROI Training and Validation Loss"
)
axis.set_xlabel("Global Epoch")
axis.set_ylabel("Loss")
axis.legend()
axis.grid(True, alpha=0.25)

figure.tight_layout()

saved_figures.append(
    save_figure(
        figure,
        "training_validation_loss_curve.png",
    )
)


# ------------------------------------------------------------
# 2. Validation metrics
# ------------------------------------------------------------

figure, axis = plt.subplots(
    figsize=(10, 6)
)

axis.plot(
    history["global_epoch"],
    history["val_roc_auc"],
    color="#1f77b4",
    linewidth=2,
    label="Validation ROC-AUC",
)

axis.plot(
    history["global_epoch"],
    history["val_f1"],
    color="#ff7f0e",
    linewidth=2,
    label="Validation F1",
)

axis.plot(
    history["global_epoch"],
    history["val_balanced_accuracy"],
    color="#2ca02c",
    linewidth=2,
    label="Validation Balanced Accuracy",
)

axis.set_title(
    "EfficientNet-B0 Mouth ROI Validation Metrics"
)
axis.set_xlabel("Global Epoch")
axis.set_ylabel("Score")
axis.set_ylim(0, 1)
axis.legend()
axis.grid(True, alpha=0.25)

figure.tight_layout()

saved_figures.append(
    save_figure(
        figure,
        "validation_metrics_curve.png",
    )
)


y_test = test_output["labels"]
p_test = test_output["probabilities"]


# ------------------------------------------------------------
# 3. Test ROC curve
# ------------------------------------------------------------

fpr, tpr, _ = roc_curve(
    y_test,
    p_test,
)

roc_auc = roc_auc_score(
    y_test,
    p_test,
)

figure, axis = plt.subplots(
    figsize=(10, 6)
)

axis.plot(
    fpr,
    tpr,
    color="#1f77b4",
    linewidth=2,
    label=(
        "EfficientNet-B0 Mouth ROI "
        f"(AUC={roc_auc:.3f})"
    ),
)

axis.plot(
    [0, 1],
    [0, 1],
    color="#7f7f7f",
    linestyle="--",
    label="Chance",
)

axis.set_title(
    "EfficientNet-B0 Mouth ROI Test ROC Curve"
)
axis.set_xlabel("False Positive Rate")
axis.set_ylabel("True Positive Rate")
axis.legend()
axis.grid(True, alpha=0.25)

figure.tight_layout()

saved_figures.append(
    save_figure(
        figure,
        "test_roc_curve.png",
    )
)


# ------------------------------------------------------------
# 4. Test precision-recall curve
# ------------------------------------------------------------

precision_curve, recall_curve, _ = (
    precision_recall_curve(
        y_test,
        p_test,
    )
)

average_precision = average_precision_score(
    y_test,
    p_test,
)

figure, axis = plt.subplots(
    figsize=(10, 6)
)

axis.plot(
    recall_curve,
    precision_curve,
    color="#ff7f0e",
    linewidth=2,
    label=(
        "EfficientNet-B0 Mouth ROI "
        f"(AP={average_precision:.3f})"
    ),
)

axis.set_title(
    "EfficientNet-B0 Mouth ROI Test Precision-Recall Curve"
)
axis.set_xlabel("Recall")
axis.set_ylabel("Precision")
axis.set_xlim(0, 1)
axis.set_ylim(0, 1)
axis.legend()
axis.grid(True, alpha=0.25)

figure.tight_layout()

saved_figures.append(
    save_figure(
        figure,
        "test_precision_recall_curve.png",
    )
)


# ------------------------------------------------------------
# 5. Test confusion matrix
# ------------------------------------------------------------

matrix = confusion_matrix(
    y_test,
    (
        p_test >= best_threshold
    ).astype(int),
    labels=[0, 1],
)

figure, axis = plt.subplots(
    figsize=(8, 8)
)

image = axis.imshow(
    matrix,
    cmap="Blues",
)

figure.colorbar(
    image,
    ax=axis,
)

axis.set_title(
    "EfficientNet-B0 Mouth ROI Test Confusion Matrix\n"
    f"Threshold={best_threshold:.3f}"
)

axis.set_xlabel("Predicted Label")
axis.set_ylabel("True Label")

axis.set_xticks(
    [0, 1],
    labels=["Real", "Fake"],
)

axis.set_yticks(
    [0, 1],
    labels=["Real", "Fake"],
)

matrix_limit = matrix.max() / 2

for row_index in range(2):
    for column_index in range(2):
        value = matrix[
            row_index,
            column_index,
        ]

        text_color = (
            "white"
            if value > matrix_limit
            else "black"
        )

        axis.text(
            column_index,
            row_index,
            str(value),
            ha="center",
            va="center",
            fontsize=14,
            color=text_color,
        )

figure.tight_layout()

saved_figures.append(
    save_figure(
        figure,
        "test_confusion_matrix.png",
    )
)


# ------------------------------------------------------------
# 6. Validation threshold analysis
# ------------------------------------------------------------

figure, axis = plt.subplots(
    figsize=(10, 6)
)

axis.plot(
    threshold_table["threshold"],
    threshold_table["f1"],
    color="#ff7f0e",
    linewidth=2,
    label="Validation F1",
)

axis.plot(
    threshold_table["threshold"],
    threshold_table["balanced_accuracy"],
    color="#2ca02c",
    linewidth=2,
    label="Validation Balanced Accuracy",
)

axis.plot(
    threshold_table["threshold"],
    threshold_table["specificity"],
    color="#9467bd",
    linewidth=2,
    label="Validation Specificity",
)

axis.axvline(
    best_threshold,
    color="#d62728",
    linestyle="--",
    linewidth=2,
    label=f"Selected={best_threshold:.3f}",
)

axis.set_title(
    "EfficientNet-B0 Mouth ROI Validation Threshold Analysis"
)
axis.set_xlabel("Threshold")
axis.set_ylabel("Score")
axis.set_ylim(0, 1)
axis.legend()
axis.grid(True, alpha=0.25)

figure.tight_layout()

saved_figures.append(
    save_figure(
        figure,
        "validation_threshold_analysis.png",
    )
)


print("=" * 72)
print("600 DPI FIGURES SAVED")
print("=" * 72)
print("Figure count :", len(saved_figures))
print("Figure folder:", DIRS["figures"])

for figure_path in saved_figures:
    with Image.open(figure_path) as image:
        print(
            f"- {figure_path.name}: "
            f"{image.size[0]}x{image.size[1]} pixels"
        )

600 DPI FIGURES SAVED
Figure count : 6
Figure folder: /content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar/20260808_1257_mouth_efficientnet_b0_seed42/figures
- training_validation_loss_curve.png: 5938x3525 pixels
- validation_metrics_curve.png: 5937x3525 pixels
- test_roc_curve.png: 5937x3525 pixels
- test_precision_recall_curve.png: 5937x3525 pixels
- test_confusion_matrix.png: 4546x4740 pixels
- validation_threshold_analysis.png: 5937x3525 pixels


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [30]:
# ============================================================
# CELL 23 — RELOAD / INFERENCE QUALITY GATE
# ============================================================

def inference_reload_test() -> dict:

    # Modeli pretrained ağırlık indirmeden sıfırdan oluştur.
    reloaded_model = (
        build_model(pretrained=False)
        .to(DEVICE)
    )

    # Eğitilmiş en iyi checkpoint'i yükle.
    checkpoint = torch.load(
        FINAL_BEST_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    reloaded_model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    reloaded_model.to(DEVICE)
    reloaded_model.eval()

    # Test setinden küçük bir batch al.
    batch = next(
        iter(loaders["test"])
    )

    sample_count = min(
        4,
        len(batch["image"]),
    )

    images = (
        batch["image"][:sample_count]
        .to(DEVICE)
    )

    with torch.no_grad():
        probabilities = torch.sigmoid(
            reloaded_model(images)
            .squeeze(1)
            .float()
        )

    if probabilities.ndim != 1:
        raise RuntimeError(
            "Unexpected inference output shape: "
            f"{tuple(probabilities.shape)}"
        )

    if not torch.isfinite(probabilities).all():
        raise FloatingPointError(
            "Reloaded model produced "
            "non-finite probabilities."
        )

    if (
        (
            probabilities < 0
        )
        |
        (
            probabilities > 1
        )
    ).any():
        raise RuntimeError(
            "Reloaded model produced probabilities "
            "outside [0, 1]."
        )

    predictions = (
        probabilities >= best_threshold
    ).int()

    return {
        "status": "PASSED",
        "model": "EfficientNet-B0",
        "roi_region": "mouth",
        "checkpoint": str(
            FINAL_BEST_CHECKPOINT
        ),
        "threshold": float(
            best_threshold
        ),
        "n_samples": int(
            len(probabilities)
        ),
        "probabilities": (
            probabilities
            .detach()
            .cpu()
            .numpy()
            .astype(float)
            .tolist()
        ),
        "predictions": (
            predictions
            .detach()
            .cpu()
            .numpy()
            .astype(int)
            .tolist()
        ),
    }


inference_test = inference_reload_test()

atomic_json_dump(
    inference_test,
    DIRS["metrics"]
    / "inference_reload_test.json",
)

print("=" * 72)
print("RELOAD / INFERENCE QUALITY GATE COMPLETED")
print("=" * 72)
print(
    json.dumps(
        inference_test,
        indent=2,
    )
)

RELOAD / INFERENCE QUALITY GATE COMPLETED
{
  "status": "PASSED",
  "model": "EfficientNet-B0",
  "roi_region": "mouth",
  "checkpoint": "/content/drive/MyDrive/AISC C\u0327al\u0131s\u0327malar/Deneyler/Dilara/Deney 1/Sonuc\u0327lar/20260808_1257_mouth_efficientnet_b0_seed42/checkpoints/finetune/best.ckpt",
  "threshold": 0.19,
  "n_samples": 4,
  "probabilities": [
    0.8178461790084839,
    0.926552414894104,
    0.6797011494636536,
    0.9244805574417114
  ],
  "predictions": [
    1,
    1,
    1,
    1
  ]
}


In [31]:
# ============================================================
# CELL 24 — FINAL MANIFEST + RUN SUMMARY + COMPLETE MARKER
# ============================================================

if inference_test.get("status") != "PASSED":
    raise RuntimeError(
        "Inference reload quality gate did not pass."
    )


def build_output_manifest(
    root: Path,
) -> pd.DataFrame:

    rows = []

    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue

        # Varsa geçici dosyaları manifest dışında bırak.
        if path.suffix.endswith(".tmp"):
            continue

        size = path.stat().st_size

        # 50 MB ve altındaki dosyaların SHA-256 özetini al.
        digest = (
            sha256_file(path)
            if size <= 50 * 1024 * 1024
            else ""
        )

        rows.append({
            "relative_path": str(
                path.relative_to(root)
            ),
            "size_bytes": int(size),
            "sha256": digest,
        })

    return pd.DataFrame(rows)


# Çıktı manifestini oluştur.
manifest = build_output_manifest(
    RUN_DIR
)

atomic_csv_dump(
    manifest,
    RUN_DIR / "output_manifest.csv",
)


run_summary = {
    "run_id": RUN_ID,
    "run_schema_version": RUN_SCHEMA_VERSION,

    "experiment": (
        "EfficientNet-B0 Mouth ROI "
        "Transfer Learning Baseline"
    ),

    "model": "torchvision EfficientNet-B0",

    "pretrained_weights": (
        "EfficientNet_B0_Weights.DEFAULT"
    ),

    "roi_region": "mouth",
    "input": "mouth ROI images",

    "image_preprocessing": (
        "224x224 RGB + ImageNet normalization"
    ),

    "image_size": CONFIG.image_size,
    "seed": CONFIG.seed,
    "device": str(DEVICE),

    "metadata_path": str(METADATA_PATH),
    "roi_root": str(ROI_ROOT),
    "results_root": str(RESULTS_ROOT),
    "run_dir": str(RUN_DIR),

    "split_counts": (
        data_accounting["split_class_counts"]
    ),

    "validation_selected_threshold": float(
        best_threshold
    ),

    "validation_metrics": (
        validation_selected_metrics
    ),

    "final_test_metrics": test_metrics,

    "inference_reload_test": (
        inference_test["status"]
    ),

    "best_model_checkpoint": str(
        FINAL_BEST_CHECKPOINT
    ),

    "figure_resolution_dpi": 600,
    "figure_count": 6,

    "output_file_count": int(
        len(manifest)
    ),
}


# Deney özetini kaydet.
atomic_json_dump(
    run_summary,
    RUN_DIR / "run_summary.json",
)


# Deneyin eksiksiz bittiğini belirten işaret dosyası.
atomic_json_dump(
    {
        "status": "COMPLETE",
        "run_id": RUN_ID,
        "completed_at": (
            datetime.now().isoformat()
        ),
    },
    RUN_DIR / RUN_COMPLETE_FILE,
)


print("=" * 80)
print("EXPERIMENT COMPLETE")
print("=" * 80)

print(
    json.dumps(
        run_summary,
        indent=2,
        ensure_ascii=False,
        default=str,
    )
)

print()
print("FINAL OUTPUT DIRECTORY:")
print(RUN_DIR)

print()
print("IMPORTANT OUTPUTS:")
print("- Best model :", FINAL_BEST_CHECKPOINT)
print(
    "- Test metrics:",
    DIRS["metrics"] / "final_test_metrics.csv",
)
print(
    "- Predictions :",
    DIRS["predictions"] / "test_predictions.csv",
)
print(
    "- Figures     :",
    DIRS["figures"],
)
print(
    "- Run summary :",
    RUN_DIR / "run_summary.json",
)
print(
    "- Manifest    :",
    RUN_DIR / "output_manifest.csv",
)

EXPERIMENT COMPLETE
{
  "run_id": "20260808_1257_mouth_efficientnet_b0_seed42",
  "run_schema_version": 4,
  "experiment": "EfficientNet-B0 Mouth ROI Transfer Learning Baseline",
  "model": "torchvision EfficientNet-B0",
  "pretrained_weights": "EfficientNet_B0_Weights.DEFAULT",
  "roi_region": "mouth",
  "input": "mouth ROI images",
  "image_preprocessing": "224x224 RGB + ImageNet normalization",
  "image_size": 224,
  "seed": 42,
  "device": "cpu",
  "metadata_path": "/content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output/metadata.csv",
  "roi_root": "/content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Ağız/mouth_roi_output",
  "results_root": "/content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar",
  "run_dir": "/content/drive/MyDrive/AISC Çalışmalar/Deneyler/Dilara/Deney 1/Sonuçlar/20260808_1257_mouth_efficientnet_b0_seed42",
  "split_counts": {
    "train": {
      "fake": 1192,
      "real": 1197
    },
    "v